# Compact continuous explanations

Can additive functions or a shallow tree describe frozen factor activations? Original rules, cutoffs and patient splits remain fixed. Reference fitting and tuning precede a private freeze; evaluation never chooses models. These are exploratory associations, not clinical or causal concepts.


In [1]:
import os, math, time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from joblib import Parallel, delayed
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor

MODE = os.environ.get("CMR_MODE", "check")
BRANCH = "explanations"
SEED = 42
FEATURE_COUNTS = (3, 5, 8)
ALPHAS = (.1, 1., 10.)
LEAF_COUNTS = (3, 5, 8)
DRAW_SEEDS = tuple(range(3001, 3501))
TIE = 1e-6
CONTINUOUS_FIELDS = ("concept", "reference", "year", "lag", "aliases", "model", "kind", "selected", "stratum", "n", "high_prevalence", "out_of_range", "features", "leaves", "status", "mae", "mse", "normalized_delta", "retained_reduction", "mae_low", "mae_high", "undefined_fraction", "normalized_delta_low", "normalized_delta_high")
CONTINUOUS_FIELDS += ("normalized_delta_undefined_fraction", "mse_low", "mse_high", "mse_undefined_fraction", "retained_reduction_low", "retained_reduction_high", "retained_reduction_undefined_fraction")
CONTINUOUS_FIELDS += ("comparison_status",)
BINARY_FIELDS = ("concept", "reference", "year", "model", "decision", "status", "n", "positive", "negative", "tp", "fp", "fn", "precision", "recall", "f2", "coverage", "lift", "threshold", "threshold_search_size") + tuple(name+suffix for name in ("precision", "recall", "f2", "coverage", "lift") for suffix in ("_low", "_high", "_undefined_fraction", "_delta_rule", "_delta_rule_low", "_delta_rule_high"))
ATTEMPT_FIELDS = ("concept", "reference", "model", "kind", "status", "error", "features", "alpha", "leaves", "tune_mae", "tune_high_mae", "threshold", "threshold_status", "threshold_search_size", "tune_f2")
COVERAGE_FIELDS = ("concept", "reference", "year", "status", "fit_n", "tune_n", "evaluation_n", "attempts", "aliases")
CURVE_FIELDS = ("concept", "feature", "x", "contribution", "intercept", "coefficient_linear", "knot_1", "knot_2", "knot_3", "coefficient_hinge_1", "coefficient_hinge_2", "coefficient_hinge_3")
LEAF_FIELDS = ("concept", "reference", "year", "leaf", "conditions", "fit_support", "fit_activation", "evaluation_support", "evaluation_activation", "evaluation_mae")
YEARLY_FIELDS = ("concept", "reference", "year", "status", "n", "target_positive", "selected_positive", "tp", "fp", "fn", "precision", "recall", "f2", "overlap", "prevalence", "coverage", "lift")
RULE_FIELDS = ("concept", "reference", "year", "rule", "clause", "feature", "operator", "value", "source")
CTX = None
if MODE not in {"check", "smoke", "develop", "evaluate"}:
    raise ValueError("Unknown CMR_MODE")
if MODE != "check":
    from cmr_common import setup_run, iter_units, freeze_write, freeze_read, publish, patient_weights, finish_run
    CTX = setup_run(BRANCH)
print(f"Mode: {MODE}; three feature limits, three ridge penalties, three tree limits; seed {SEED}")

Mode: evaluate; three feature limits, three ridge penalties, three tree limits; seed 42


## Fitting-only feature functions
Each nonconstant feature contributes a line plus hinges at fitting quartiles. Univariate squared-error reduction ranks features. Fitting rows alone determine knots, means, scales and ranges; no interactions enter the additive model.


In [2]:
def feature_basis(x, knots):
    x = np.asarray(x, dtype=float)
    return np.column_stack([x, *[np.maximum(x-k, 0.) for k in knots]])

def rank_features(X, y):
    X, y = np.asarray(X, float), np.asarray(y, float)
    total = np.square(y-y.mean()).sum()
    metadata = []
    for j in range(X.shape[1]):
        x = X[:, j]
        if np.ptp(x) == 0:
            continue
        knots = np.unique(np.quantile(x, [.25, .5, .75]))
        basis = feature_basis(x, knots)
        design = np.column_stack([np.ones(len(x)), basis])
        fitted = design @ np.linalg.lstsq(design, y, rcond=None)[0]
        metadata.append({"feature": j, "knots": knots, "minimum": x.min(), "maximum": x.max(),
                         "improvement": float(total-np.square(y-fitted).sum())})
    return sorted(metadata, key=lambda m: (-m["improvement"], m["feature"]))

print("Transforms: fit-only quartile hinges; constants excluded; feature-index ties")

Transforms: fit-only quartile hinges; constants excluded; feature-index ties


In [3]:
def additive_basis(X, metadata):
    blocks = [feature_basis(X[:, m["feature"]], m["knots"]) for m in metadata]
    return np.column_stack(blocks) if blocks else np.empty((len(X), 0))

def fit_additive(X, y, ranking, count, alpha):
    metadata = ranking[:count]
    if not metadata:
        raise ValueError("No nonconstant fitting features")
    basis = additive_basis(X, metadata)
    center, scale = basis.mean(0), basis.std(0)
    scale[scale == 0] = 1.
    model = Ridge(alpha=alpha).fit((basis-center)/scale, y)
    return {"kind": "additive", "model": model, "metadata": metadata,
            "center": center, "scale": scale, "features": len(metadata), "alpha": alpha}

def predict_model(bundle, X):
    if bundle["kind"] == "additive":
        X = (additive_basis(X, bundle["metadata"])-bundle["center"])/bundle["scale"]
    predicted = bundle["model"].predict(X)
    if not np.isfinite(predicted).all():
        raise ValueError("Nonfinite continuous predictions")
    return predicted

print("Joint ridge preserves centered feature contributions; native predictions retained")

Joint ridge preserves centered feature contributions; native predictions retained


## Binary decisions retain the original target
Original-high means actual activation strictly above its saved cutoff. A second decision threshold maximizes tuning F2 under original precision and lift constraints. No feasible threshold stays unsupported.


In [4]:
def binary_metrics(target, decision, weight=None):
    target, decision = np.asarray(target, bool), np.asarray(decision, bool)
    w = np.ones(len(target)) if weight is None else np.asarray(weight, float)
    n, positives, selected = w.sum(), w[target].sum(), w[decision].sum()
    tp = w[target & decision].sum(); fp = selected-tp; fn = positives-tp
    precision = tp/selected if selected else None
    recall = tp/positives if positives else None
    prevalence = positives/n if n else None
    denominator = 5*tp+4*fn+fp
    return {"n": float(n), "positive": float(positives), "negative": float(n-positives),
            "tp": float(tp), "fp": float(fp), "fn": float(fn),
            "precision": precision, "recall": recall,
            "f2": 5*tp/denominator if denominator else None,
            "coverage": selected/n if n else None,
            "lift": precision/prevalence if precision is not None and prevalence else None}

def tune_threshold(target, predicted):
    target, predicted = np.asarray(target, bool), np.asarray(predicted, float)
    if not len(target) or not target.any() or not np.isfinite(predicted).all():
        return {"status": "unsupported", "threshold": None, "search_size": 0}
    unique = np.unique(predicted)
    candidates = np.r_[np.nextafter(unique[0], -np.inf), unique, np.nextafter(unique[-1], np.inf)]
    best = None
    for threshold in candidates:
        metric = binary_metrics(target, predicted > threshold)
        if metric["precision"] is None or metric["precision"] < .5 or metric["lift"] < 1.5:
            continue
        if best is None or metric["f2"] > best["f2"]+TIE or (abs(metric["f2"]-best["f2"]) <= TIE and
            (metric["precision"], threshold) > (best["precision"], best["threshold"])):
            best = {**metric, "threshold": float(threshold)}
    return {"status": "measured" if best else "unsupported", "threshold": best["threshold"] if best else None,
            "search_size": len(candidates), "tune_support": len(target), "positive": int(target.sum()),
            "tune_f2": best["f2"] if best else None}

print("Strict >; at most n_tune+2 thresholds; precision >= .5 and lift >= 1.5")

Strict >; at most n_tune+2 thresholds; precision >= .5 and lift >= 1.5


## Fixed model panel
Two constants, historical histogram booster, nine additive candidates, three shallow trees and fixed-alpha ridge fit actual activations. Independent sklearn fits use CPU workers. Tuning MAE chooses complexity; ties favor smaller models then stronger additive regularization. Every attempt remains visible.


In [5]:
def fit_candidate(name, kind, X, y, ranking, setting):
    # pragmatic: sklearn fits are CPU-only; parallel workers serve this bounded tabular panel.
    try:
        if kind == "additive":
            bundle = fit_additive(X, y, ranking, *setting)
        else:
            if kind in ("constant_mean", "constant_median"):
                model = DummyRegressor(strategy="mean" if kind == "constant_mean" else "median")
            elif kind == "nonlinear":
                model = HistGradientBoostingRegressor(max_iter=50, max_leaf_nodes=7,
                    min_samples_leaf=10, l2_regularization=1., early_stopping=False, random_state=SEED)
            elif kind == "ridge":
                model = make_pipeline(StandardScaler(), Ridge(alpha=1.))
            elif kind == "tree":
                model = DecisionTreeRegressor(criterion="squared_error", max_leaf_nodes=setting,
                    max_depth=3, min_samples_leaf=max(10, math.ceil(.1*len(y))), random_state=SEED)
            else:
                raise ValueError("Unknown fixed model kind")
            model.fit(X, y)
            bundle = {"kind": kind, "model": model, "leaves": model.get_n_leaves() if kind == "tree" else None}
        predict_model(bundle, X)
        return name, bundle, None
    except (ValueError, FloatingPointError, np.linalg.LinAlgError) as error:
        return name, None, type(error).__name__

print("Panel: 2 constants + 1 nonlinear + 9 additive + 3 tree + 1 ridge = 16 fits; separate screening")

Panel: 2 constants + 1 nonlinear + 9 additive + 3 tree + 1 ridge = 16 fits; separate screening


In [6]:
def choose_complexity(attempts, kind):
    eligible = [a for a in attempts if a["status"] == "measured" and a["kind"] == kind]
    if not eligible:
        return None
    minimum = min(a["tune_mae"] for a in eligible)
    tied = [a for a in eligible if a["tune_mae"] <= minimum+TIE]
    if kind == "additive":
        return min(tied, key=lambda a: (a["features"], -a["alpha"], a["model"]))["model"]
    return min(tied, key=lambda a: (a["leaves"], a["model"]))["model"]

def fit_panel(X_fit, y_fit, X_tune, y_tune, cutoff):
    if not len(y_fit) or not len(y_tune):
        return {"status": "unsupported", "models": {}, "attempts": [], "selected": {}}
    if not all(np.isfinite(a).all() for a in (X_fit, X_tune, y_fit, y_tune)):
        raise ValueError("Finite fitting and tuning inputs required")
    ranking = rank_features(X_fit, y_fit)
    settings = [(name, name, None) for name in ("constant_mean", "constant_median", "nonlinear", "ridge")]
    settings += [(f"additive_{k}_{alpha:g}", "additive", (k, alpha)) for k in FEATURE_COUNTS for alpha in ALPHAS]
    settings += [(f"tree_{leaves}", "tree", leaves) for leaves in LEAF_COUNTS]
    fitted = Parallel(n_jobs=-1, prefer="threads")(
        delayed(fit_candidate)(name, kind, X_fit, y_fit, ranking, setting) for name, kind, setting in settings)
    models, attempts, thresholds = {}, [], {}
    high = np.asarray(y_tune) > cutoff
    for name, bundle, error in fitted:
        kind = next(kind for candidate, kind, _ in settings if candidate == name)
        if error:
            attempts.append({"model": name, "kind": kind, "status": "failed", "error": error})
            continue
        predicted = predict_model(bundle, X_tune)
        attempts.append({"model": name, "kind": kind, "status": "measured", "error": None,
            "features": bundle.get("features"), "alpha": bundle.get("alpha"), "leaves": bundle.get("leaves"),
            "tune_mae": float(np.abs(y_tune-predicted).mean()),
            "tune_high_mae": float(np.abs(y_tune[high]-predicted[high]).mean()) if high.any() else None})
        models[name] = bundle
        thresholds[name] = tune_threshold(high, predicted)
        attempts[-1].update(threshold=thresholds[name]["threshold"], threshold_status=thresholds[name]["status"],
                            threshold_search_size=thresholds[name]["search_size"], tune_f2=thresholds[name].get("tune_f2"))
    selected = {kind: choose_complexity(attempts, kind) for kind in ("additive", "tree")}
    return {"status": "measured", "models": models, "attempts": attempts,
            "selected": selected, "thresholds": thresholds, "ranking": ranking,
            "fit_rows": len(y_fit), "tune_rows": len(y_tune), "cutoff": float(cutoff)}

print("Development accepts fitting/tuning arrays only; evaluation is not a model-selection input")

Development accepts fitting/tuning arrays only; evaluation is not a model-selection input


## Conditional patient uncertainty
Shared patient draws keep years and overlapping concepts together. Full, zero, positive and original-high strata use actual activation. Conditional intervals hold models fixed. Empty strata and zero normalization denominators remain undefined.


In [7]:
def interval(values):
    values = np.asarray([np.nan if v is None else v for v in values], float)
    finite = values[np.isfinite(values)]
    return {"low": float(np.quantile(finite, .025)) if len(finite) else None,
            "high": float(np.quantile(finite, .975)) if len(finite) else None,
            "undefined_fraction": float(1-len(finite)/len(values)) if len(values) else 1.}

def weighted_mean(values, weights):
    device = CTX["device"] if CTX else "cpu"
    tensor_weights = torch.as_tensor(weights, dtype=torch.float64, device=device)
    tensor_values = torch.as_tensor(values, dtype=torch.float64, device=device)
    denominator = tensor_weights.sum(dim=-1).cpu().numpy()
    numerator = (tensor_weights @ tensor_values).cpu().numpy()
    return np.divide(numerator, denominator, out=np.full_like(numerator, np.nan, dtype=float), where=denominator>0)

def benchmark_comparison(truth, predicted, nonlinear, constant, weights):
    if not len(truth) or nonlinear is None or constant is None:
        return {"normalized_delta": None, "retained_reduction": None,
                "normalized_delta_low": None, "normalized_delta_high": None, "normalized_delta_undefined_fraction": 1.,
                "retained_reduction_low": None, "retained_reduction_high": None, "retained_reduction_undefined_fraction": 1.,
                "comparison_status": "undefined" if not len(truth) else "missing_dependency"}
    error, baseline, denominator = np.abs(truth-predicted), np.abs(truth-nonlinear), np.abs(truth-constant)
    mae, nonlinear_mae, constant_mae = error.mean(), baseline.mean(), denominator.mean()
    sampled_error = weighted_mean(error, weights)
    sampled_denominator = weighted_mean(denominator, weights)
    sampled_delta = weighted_mean(error-baseline, weights)
    normalized = np.divide(sampled_delta, sampled_denominator, out=np.full_like(sampled_delta, np.nan), where=sampled_denominator>0)
    delta_ci = interval(normalized)
    sampled_gain = weighted_mean(denominator-baseline, weights)
    retained = np.divide(sampled_denominator-sampled_error, sampled_gain, out=np.full_like(sampled_gain, np.nan), where=sampled_gain>0)
    retained_ci = interval(retained)
    return {"comparison_status": "measured",
            "normalized_delta": float((mae-nonlinear_mae)/constant_mae) if constant_mae>0 else None,
            "retained_reduction": float((constant_mae-mae)/(constant_mae-nonlinear_mae)) if constant_mae>nonlinear_mae else None,
            "normalized_delta_low": delta_ci["low"], "normalized_delta_high": delta_ci["high"], "normalized_delta_undefined_fraction": delta_ci["undefined_fraction"],
            "retained_reduction_low": retained_ci["low"], "retained_reduction_high": retained_ci["high"], "retained_reduction_undefined_fraction": retained_ci["undefined_fraction"]}

print("Paired benchmark comparisons require both frozen references; absent references remain missing")

Paired benchmark comparisons require both frozen references; absent references remain missing


In [8]:
def continuous_record(truth, predicted, nonlinear, constant, weights):
    comparison = benchmark_comparison(truth, predicted, nonlinear, constant, weights)
    if not len(truth):
        return {"status": "undefined", "mae": None, "mse": None,
                "mae_low": None, "mae_high": None, "undefined_fraction": 1.,
                "mse_low": None, "mse_high": None, "mse_undefined_fraction": 1., **comparison}
    absolute, squared = np.abs(truth-predicted), np.square(truth-predicted)
    mae_ci, mse_ci = interval(weighted_mean(absolute, weights)), interval(weighted_mean(squared, weights))
    return {"status": "measured", "mae": float(absolute.mean()), "mse": float(squared.mean()),
            "mae_low": mae_ci["low"], "mae_high": mae_ci["high"], "undefined_fraction": mae_ci["undefined_fraction"],
            "mse_low": mse_ci["low"], "mse_high": mse_ci["high"], "mse_undefined_fraction": mse_ci["undefined_fraction"], **comparison}

print("500 shared patient draws; no refitting; original-high is activation > saved cutoff")

500 shared patient draws; no refitting; original-high is activation > saved cutoff


In [9]:
def sampled_binary(target, decision, weights):
    target, decision = np.asarray(target, bool), np.asarray(decision, bool)
    columns = np.column_stack([target & decision, decision, target])
    device = CTX["device"] if CTX else "cpu"
    w = torch.as_tensor(weights, dtype=torch.float64, device=device)
    counts = (w @ torch.as_tensor(columns, dtype=torch.float64, device=device)).cpu().numpy()
    n = w.sum(dim=1).cpu().numpy()
    tp, selected, positives = counts.T
    def divide(top, bottom):
        return np.divide(top, bottom, out=np.full_like(top, np.nan), where=bottom>0)
    precision, prevalence = divide(tp, selected), divide(positives, n)
    return {"precision": precision, "recall": divide(tp, positives), "f2": divide(5*tp, 4*positives+selected),
            "coverage": divide(selected, n), "lift": divide(precision, prevalence)}

def binary_record(target, decision, weights, original_rule=None):
    result = binary_metrics(target, decision)
    sampled = sampled_binary(target, decision, weights)
    for name in ("precision", "recall", "f2", "coverage", "lift"):
        ci = interval(sampled[name])
        result.update({name+"_low": ci["low"], name+"_high": ci["high"], name+"_undefined_fraction": ci["undefined_fraction"]})
        if original_rule is not None:
            reference = binary_metrics(target, original_rule)
            paired = sampled_binary(target, original_rule, weights)
            delta = interval(sampled[name]-paired[name])
            result.update({name+"_delta_rule": result[name]-reference[name] if result[name] is not None and reference[name] is not None else None,
                           name+"_delta_rule_low": delta["low"], name+"_delta_rule_high": delta["high"]})
    return result

def outside_range(bundle, X):
    if bundle["kind"] != "additive" or not len(X):
        return None
    outside = np.column_stack([(X[:,m["feature"]] < m["minimum"]) | (X[:,m["feature"]] > m["maximum"]) for m in bundle["metadata"]])
    return float(outside.any(axis=1).mean())

print("Binary precision, recall, F2, coverage and lift retain support and undefined-draw fractions")

Binary precision, recall, F2, coverage and lift retain support and undefined-draw fractions


## Readable curves and tree groups
Feature curves use fixed fitting-quantile grids. Tree paths show conditions and fitted leaf activation; independent leaf errors use evaluation patients only. These aggregate descriptions contain no patient scatterplots or fitted-object exports.


In [10]:
def additive_curves(bundle, names, X_fit, concept):
    rows, offset = [], 0
    for m in bundle["metadata"]:
        feature = m["feature"]
        grid = np.unique(np.quantile(X_fit[:,feature], np.linspace(0, 1, 11)))
        block = feature_basis(grid, m["knots"])
        width = block.shape[1]
        coefficient = bundle["model"].coef_[offset:offset+width]/bundle["scale"][offset:offset+width]
        contribution = (block-bundle["center"][offset:offset+width]) @ coefficient
        for x, value in zip(grid, contribution):
            record = {"concept": concept, "feature": str(names[feature]), "x": float(x), "contribution": float(value),
                      "intercept": float(bundle["model"].intercept_), "coefficient_linear": float(coefficient[0])}
            for i, knot in enumerate(m["knots"], 1):
                record.update({f"knot_{i}": float(knot), f"coefficient_hinge_{i}": float(coefficient[i])})
            rows.append(record)
        offset += width
    return rows

def tree_groups(bundle, names, X, truth, concept, reference=None, year=None):
    tree, rows = bundle["model"].tree_, []
    leaves = bundle["model"].apply(X) if len(X) else np.array([], int)
    pending = [(0, [])]
    while pending:
        node, conditions = pending.pop()
        if tree.children_left[node] == tree.children_right[node]:
            mask = leaves == node
            rows.append({"concept": concept, "reference": reference, "year": year, "leaf": node, "conditions": " and ".join(conditions) or "all rows",
                         "fit_support": int(tree.n_node_samples[node]), "fit_activation": float(tree.value[node,0,0]),
                         "evaluation_support": int(mask.sum()), "evaluation_activation": float(truth[mask].mean()) if mask.any() else None,
                         "evaluation_mae": float(np.abs(truth[mask]-tree.value[node,0,0]).mean()) if mask.any() else None})
        else:
            label, threshold = str(names[tree.feature[node]]), tree.threshold[node]
            pending.append((tree.children_right[node], conditions+[f"{label} > {threshold:.8g}"]))
            pending.append((tree.children_left[node], conditions+[f"{label} <= {threshold:.8g}"]))
    return rows

def rule_conditions(unit, rule, year, source):
    rows = []
    for i, clause in enumerate(rule["rules"]):
        for j, condition in enumerate(clause["conditions"]):
            rows.append({"concept": unit["key"], "reference": unit["reference"], "year": year,
                         "rule": i, "clause": j, "feature": str(unit["feature_names"][condition["feature_index"]]),
                         "operator": condition["operator"], "value": float(condition["threshold"]), "source": source})
    return rows

print("Curves and leaf paths derive from frozen fitting objects; held-out errors remain independent")

Curves and leaf paths derive from frozen fitting objects; held-out errors remain independent


## Reference development and freeze
Every original rule-linked medoid enters development, including historically ineligible families. Baseline, last_3 and all_history panels fit separately; exact aliases share computation. Models, transforms, cutoffs and threshold attempts remain private in this branch's freeze.


In [11]:
BUNDLES, INPUTS, ATTEMPTS, COVERAGE = {}, {}, [], []
CURVES, LEAVES, CONDITIONS = [], [], []
if MODE in {"develop", "smoke"}:
    for unit in iter_units(CTX, BRANCH):
        fit, tune = unit["fit"], unit["tune"]
        if set(unit["patients"][fit]) & set(unit["patients"][tune]):
            raise ValueError("Fitting/tuning patient overlap")
        panel = fit_panel(unit["X"][fit], unit["activation"][fit], unit["X"][tune], unit["activation"][tune], unit["cutoff"])
        additive = panel["selected"].get("additive")
        if additive:
            panel["models"][additive]["curves"] = additive_curves(panel["models"][additive], unit["feature_names"], unit["X"][fit], unit["key"])
            CURVES.extend(panel["models"][additive]["curves"])
        tree = panel["selected"].get("tree")
        if tree:
            LEAVES.extend(tree_groups(panel["models"][tree], unit["feature_names"], unit["X"][:0], unit["activation"][:0], unit["key"], unit["reference"], unit["reference"]))
        BUNDLES[unit["key"]] = panel
        INPUTS[unit["key"]] = unit["identity"]
        CONDITIONS.extend(rule_conditions(unit, unit["rule_dict"], unit["reference"], "original_frozen_rule"))
        ATTEMPTS.extend({"concept": unit["key"], "reference": unit["reference"], **a} for a in panel["attempts"])
        COVERAGE.append({"concept": unit["key"], "reference": unit["reference"], "status": panel["status"],
                         "fit_n": len(set(unit["patients"][fit])), "tune_n": len(set(unit["patients"][tune])),
                         "attempts": len(panel["attempts"]), "aliases": ",".join(unit["aliases"])})
    if MODE == "develop":
        FREEZE_HASH = freeze_write(CTX, {"units": BUNDLES, "inputs": INPUTS})
        print("Private branch freeze:", FREEZE_HASH)
    display(pd.DataFrame(COVERAGE))
    display(pd.DataFrame(ATTEMPTS))
else:
    print("Reference fits guarded in this mode")

Reference fits guarded in this mode


## Frozen reference and later-year fidelity
All frozen candidates appear as descriptive complexity curves. Tuning alone identifies primary additive/tree estimates. Original rules remain binary comparators. Later-year data cannot modify transforms or thresholds; no activation MAE is computed from rule output.


In [12]:
def continuous_year(unit, panel, rows, weights, year, prediction):
    X, truth = unit["X"][rows], unit["activation"][rows]
    masks = {"full": np.ones(len(rows), bool), "zero": truth == 0, "positive": truth > 0, "original_high": truth > unit["cutoff"]}
    continuous = []
    for name, predicted in prediction.items():
        bundle = panel["models"][name]
        for stratum, mask in masks.items():
            continuous.append({"concept": unit["key"], "reference": unit["reference"], "year": year,
                "lag": year-unit["reference"], "aliases": ",".join(unit["aliases"]), "model": name,
                "kind": bundle["kind"], "selected": name in panel["selected"].values() or bundle["kind"] not in {"additive", "tree"},
                "stratum": stratum, "n": len(set(unit["patients"][rows][mask])), "features": bundle.get("features"), "leaves": bundle.get("leaves"),
                "high_prevalence": float(masks["original_high"].mean()), "out_of_range": outside_range(bundle, X),
                **continuous_record(truth[mask], predicted[mask], prediction["nonlinear"][mask] if "nonlinear" in prediction else None,
                    prediction["constant_median"][mask] if "constant_median" in prediction else None, weights[:,mask])})
    return continuous

print("Raw error tables retain every computable model, including when a benchmark fit failed")

Raw error tables retain every computable model, including when a benchmark fit failed


In [13]:
def binary_year(unit, panel, rows, weights, year, prediction):
    target, binary = unit["activation"][rows] > unit["cutoff"], []
    for name, predicted in prediction.items():
        threshold = panel["thresholds"][name]
        decisions = [("original_cutoff", predicted > unit["cutoff"])]
        if threshold["status"] == "measured":
            decisions.append(("tune_selected", predicted > threshold["threshold"]))
        else:
            binary.append({"concept": unit["key"], "reference": unit["reference"], "year": year,
                           "model": name, "decision": "tune_selected", "status": "unsupported"})
        for decision, mask in decisions:
            binary.append({"concept": unit["key"], "reference": unit["reference"], "year": year,
                "model": name, "decision": decision, "status": "measured", "threshold": unit["cutoff"] if decision == "original_cutoff" else threshold["threshold"],
                "threshold_search_size": threshold["search_size"], **binary_record(target, mask, weights, unit["rule_mask"][rows])})
    binary.append({"concept": unit["key"], "reference": unit["reference"], "year": year,
        "model": "original_rule", "decision": "frozen_rule", "status": "measured",
        **binary_record(target, unit["rule_mask"][rows], weights)})
    return binary

def score_year(unit, panel, rows, weights, year):
    prediction = {name: predict_model(bundle, unit["X"][rows]) for name, bundle in panel["models"].items()}
    return continuous_year(unit, panel, rows, weights, year, prediction), binary_year(unit, panel, rows, weights, year, prediction)

print("Frozen transforms, complexity, original cutoff and tuned thresholds applied unchanged")

Frozen transforms, complexity, original cutoff and tuned thresholds applied unchanged


In [14]:
def yearly_rule_record(unit, year, rows):
    fitting = np.flatnonzero((unit["years"] == year) & np.isin(unit["roles"], ["rule_discovery", "fitting"]))
    result = unit["yearly_description"](unit["X"], unit["activation"], unit["cutoff"], unit["patients"],
        fitting, rows, list(unit["feature_names"]), unit["yearly_recipe"], year, unit["family"])
    record = {"status": result["status"]}
    if "independent_fidelity" in result:
        fidelity = result["independent_fidelity"]
        mapping = {"n":"n_samples", "target_positive":"n_positive", "selected_positive":"n_selected", "tp":"true_positive", "fp":"false_positive", "fn":"false_negative", "overlap":"jaccard"}
        record.update({name:fidelity[key] for name,key in mapping.items()})
        record.update({name:fidelity[name] for name in ("precision", "recall", "f2", "prevalence", "coverage", "lift")})
        CONDITIONS.extend(rule_conditions(unit, result["rule"], year, "same_year_description"))
    return record

CONTINUOUS, BINARY, YEARLY_RULES = [], [], []
if MODE == "evaluate":
    FROZEN = freeze_read(CTX)
    BUNDLES, INPUTS = FROZEN["units"], FROZEN["inputs"]
    for unit in iter_units(CTX, BRANCH):
        if unit["key"] not in BUNDLES or unit["identity"] != INPUTS[unit["key"]]:
            raise ValueError("Explanation freeze/input identity mismatch")
        panel, evaluation = BUNDLES[unit["key"]], unit["evaluation"]
        ATTEMPTS.extend({"concept": unit["key"], "reference": unit["reference"], **a} for a in panel["attempts"])
        CONDITIONS.extend(rule_conditions(unit, unit["rule_dict"], unit["reference"], "original_frozen_rule"))
        shared_weights = patient_weights(unit["patients"][evaluation], seeds=DRAW_SEEDS)
        for year in range(unit["reference"], 2016):
            at = unit["years"][evaluation] == year
            rows = evaluation[at]
            if not len(rows):
                COVERAGE.append({"concept": unit["key"], "reference": unit["reference"], "year": year, "status": "undefined", "evaluation_n": 0})
                continue
            continuous, binary = score_year(unit, panel, rows, shared_weights[:,at], year)
            CONTINUOUS.extend(continuous); BINARY.extend(binary)
            tree = panel["selected"].get("tree")
            if tree:
                LEAVES.extend(tree_groups(panel["models"][tree], unit["feature_names"], unit["X"][rows], unit["activation"][rows], unit["key"], unit["reference"], year))
            if "yearly_description" in unit:
                YEARLY_RULES.append({"concept": unit["key"], "reference": unit["reference"], "year": year,
                                     **yearly_rule_record(unit, year, rows)})
            else:
                YEARLY_RULES.append({"concept": unit["key"], "reference": unit["reference"], "year": year,
                                     "status": "missing_dependency"})
        additive = panel["selected"].get("additive")
        if additive:
            CURVES.extend(panel["models"][additive]["curves"])
    display(pd.DataFrame(CONTINUOUS))
    display(pd.DataFrame(BINARY))
    display(pd.DataFrame(YEARLY_RULES))
else:
    print("Evaluation-role and later-year observations guarded before freeze")

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,concept,reference,year,lag,aliases,model,kind,selected,stratum,n,...,mse_undefined_fraction,comparison_status,normalized_delta,retained_reduction,normalized_delta_low,normalized_delta_high,normalized_delta_undefined_fraction,retained_reduction_low,retained_reduction_high,retained_reduction_undefined_fraction
0,"r1/2007/2007/42/42:94,135:3,215:146",2007,2007,0,"last_3,all_history,baseline",constant_mean,constant_mean,True,full,237,...,0.0,measured,0.598116,-0.349475,0.497637,0.698105,0.0,-0.768573,-0.133779,0.0
1,"r1/2007/2007/42/42:94,135:3,215:146",2007,2007,0,"last_3,all_history,baseline",constant_mean,constant_mean,True,zero,108,...,0.0,measured,4.203227,NaN,3.589044,4.759624,0.0,NaN,NaN,1.0
2,"r1/2007/2007/42/42:94,135:3,215:146",2007,2007,0,"last_3,all_history,baseline",constant_mean,constant_mean,True,positive,129,...,0.0,measured,0.377125,0.337053,0.271963,0.457477,0.0,0.253525,0.463223,0.0
3,"r1/2007/2007/42/42:94,135:3,215:146",2007,2007,0,"last_3,all_history,baseline",constant_mean,constant_mean,True,original_high,21,...,0.0,measured,0.584386,0.185160,0.537468,0.632702,0.0,0.172030,0.200095,0.0
4,"r1/2007/2007/42/42:94,135:3,215:146",2007,2007,0,"last_3,all_history,baseline",constant_median,constant_median,True,full,237,...,0.0,measured,0.443221,0.000000,0.330267,0.523312,0.0,0.000000,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46907,"r1/2013/2013/42/42:273,135:220,215:35",2013,2015,2,baseline,tree_5,tree,True,original_high,248,...,0.0,measured,0.019355,0.903061,-0.000736,0.039584,0.0,0.805449,1.003737,0.0
46908,"r1/2013/2013/42/42:273,135:220,215:35",2013,2015,2,baseline,tree_8,tree,False,full,293,...,0.0,measured,0.008104,0.958763,-0.012569,0.027411,0.0,0.861519,1.064357,0.0
46909,"r1/2013/2013/42/42:273,135:220,215:35",2013,2015,2,baseline,tree_8,tree,False,zero,0,...,1.0,undefined,NaN,NaN,NaN,NaN,1.0,NaN,NaN,1.0
46910,"r1/2013/2013/42/42:273,135:220,215:35",2013,2015,2,baseline,tree_8,tree,False,positive,293,...,0.0,measured,0.008104,0.958763,-0.012569,0.027411,0.0,0.861519,1.064357,0.0


,concept,reference,year,model,decision,status,threshold,threshold_search_size,n,positive,...,coverage_undefined_fraction,coverage_delta_rule,coverage_delta_rule_low,coverage_delta_rule_high,lift_low,lift_high,lift_undefined_fraction,lift_delta_rule,lift_delta_rule_low,lift_delta_rule_high
0,"r1/2007/2007/42/42:94,135:3,215:146",2007,2007,constant_mean,tune_selected,unsupported,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"r1/2007/2007/42/42:94,135:3,215:146",2007,2007,constant_mean,original_cutoff,measured,1.817271,3.0,237.0,21.0,...,0.0,-0.126582,-0.171494,-0.086549,NaN,NaN,1.0,NaN,NaN,NaN
2,"r1/2007/2007/42/42:94,135:3,215:146",2007,2007,constant_median,tune_selected,unsupported,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"r1/2007/2007/42/42:94,135:3,215:146",2007,2007,constant_median,original_cutoff,measured,1.817271,3.0,237.0,21.0,...,0.0,-0.126582,-0.171494,-0.086549,NaN,NaN,1.0,NaN,NaN,NaN
4,"r1/2007/2007/42/42:94,135:3,215:146",2007,2007,nonlinear,original_cutoff,measured,1.817271,82.0,237.0,21.0,...,0.0,-0.071730,-0.103329,-0.040643,4.833487,12.180979,0.0,0.665568,-1.723099,3.675232
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24184,"r1/2013/2013/42/42:273,135:220,215:35",2013,2015,tree_5,tune_selected,unsupported,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24185,"r1/2013/2013/42/42:273,135:220,215:35",2013,2015,tree_5,original_cutoff,measured,0.700169,6.0,293.0,248.0,...,0.0,0.040956,0.019638,0.063170,1.131175,1.241864,0.0,0.000000,0.000000,0.000000
24186,"r1/2013/2013/42/42:273,135:220,215:35",2013,2015,tree_8,tune_selected,unsupported,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24187,"r1/2013/2013/42/42:273,135:220,215:35",2013,2015,tree_8,original_cutoff,measured,0.700169,6.0,293.0,248.0,...,0.0,0.040956,0.019638,0.063170,1.131175,1.241864,0.0,0.000000,0.000000,0.000000


,concept,reference,year,status,n,target_positive,selected_positive,tp,fp,fn,overlap,precision,recall,f2,prevalence,coverage,lift
0,"r1/2007/2007/42/42:94,135:3,215:146",2007,2007,available,237.0,21.0,0.0,0.0,0.0,21.0,0.000000,NaN,0.000000,0.000000,0.088608,0.000000,0.000000
1,"r1/2007/2007/42/42:94,135:3,215:146",2007,2008,insufficient positive yearly fit/tune support,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"r1/2007/2007/42/42:94,135:3,215:146",2007,2009,insufficient positive yearly fit/tune support,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"r1/2007/2007/42/42:94,135:3,215:146",2007,2010,insufficient positive yearly fit/tune support,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"r1/2007/2007/42/42:94,135:3,215:146",2007,2011,insufficient positive yearly fit/tune support,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
728,"r1/2013/2013/42/42:212,135:276,215:238",2013,2014,insufficient positive yearly fit/tune support,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
729,"r1/2013/2013/42/42:212,135:276,215:238",2013,2015,insufficient positive yearly fit/tune support,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
730,"r1/2013/2013/42/42:273,135:220,215:35",2013,2013,available,322.0,47.0,22.0,17.0,5.0,30.0,0.326923,0.772727,0.361702,0.404762,0.145963,0.068323,5.294004
731,"r1/2013/2013/42/42:273,135:220,215:35",2013,2014,available,324.0,197.0,42.0,41.0,1.0,156.0,0.207071,0.976190,0.208122,0.246988,0.608025,0.129630,1.605511


## Findings
Full-cohort and original-high errors answer different questions. Compare normalized paired errors, retained benchmark improvement, complexity and strong-activation fidelity together. Support and undefined cases stay visible. Intervals are conditional and pointwise; overlapping families provide descriptive summaries.


In [15]:
display(pd.DataFrame(CURVES))
display(pd.DataFrame(LEAVES))
display(pd.DataFrame(CONDITIONS))
if CURVES:
    frame = pd.DataFrame(CURVES)
    first = frame[frame.concept == frame.concept.iloc[0]]
    for name, group in first.groupby("feature"):
        plt.plot(group.x, group.contribution, label=name)
    plt.xlabel("Feature count"); plt.ylabel("Centered activation contribution"); plt.legend(); plt.show()
if CONTINUOUS:
    frame = pd.DataFrame(CONTINUOUS)
    summary = frame[(frame.year == frame.reference) & frame.selected].groupby(["aliases", "stratum", "kind"], dropna=False).agg(
        concepts=("concept", "nunique"), median_mae=("mae", "median"), median_normalized_delta=("normalized_delta", "median")).reset_index()
    summary = summary.astype(object).where(summary.notna(), None)
    display(publish(CTX, summary.to_dict("records"), tuple(summary.columns)))
    view = frame[frame.selected & (frame.stratum == "full")]
    for name, group in view.groupby("kind"):
        plt.plot(group.groupby("lag").mae.median(), label=name)
    plt.xlabel("Years from frozen reference"); plt.ylabel("Median full-cohort MAE"); plt.legend(); plt.show()
    first_key = sorted(frame.concept.unique())[0]
    complexity = frame[(frame.concept == first_key) & (frame.year == frame.reference) & frame.kind.isin(["additive", "tree"])]
    for stratum, group in complexity[complexity.stratum.isin(["full", "original_high"])].groupby("stratum"):
        ordered = group.sort_values("model")
        plt.plot(ordered.model, ordered.mae, marker="o", label=stratum)
    plt.xticks(rotation=65); plt.ylabel("Frozen evaluation MAE"); plt.title(first_key); plt.legend(); plt.show()
else:
    print("No evaluation findings in this mode; empty support is not zero error")
if ATTEMPTS:
    attempts = pd.DataFrame(ATTEMPTS)
    for kind, axis in (("additive", "features"), ("tree", "leaves")):
        curve = attempts[(attempts.kind == kind) & (attempts.status == "measured")]
        for name, group in curve.groupby("concept"):
            plt.plot(group.groupby(axis).tune_mae.min(), alpha=.3)
        plt.xlabel(axis); plt.ylabel("Reference tune MAE"); plt.title(kind); plt.show()
if MODE != "check":
    for records, fields in ((ATTEMPTS, ATTEMPT_FIELDS), (COVERAGE, COVERAGE_FIELDS), (CONTINUOUS, CONTINUOUS_FIELDS),
                            (BINARY, BINARY_FIELDS), (CURVES, CURVE_FIELDS), (LEAVES, LEAF_FIELDS), (YEARLY_RULES, YEARLY_FIELDS), (CONDITIONS, RULE_FIELDS)):
        if records:
            publish(CTX, records, fields)
print("Report: frozen candidates, support, high-stratum errors, decisions, curves and tree groups")

,concept,feature,x,contribution,intercept,coefficient_linear,knot_1,coefficient_hinge_1,knot_2,coefficient_hinge_2,knot_3,coefficient_hinge_3
0,"r1/2007/2007/42/42:94,135:3,215:146",DIAGN_N180,0.0,-0.190172,0.390020,0.029641,0.0,0.029641,2.0,0.019456,9.5,-0.264110
1,"r1/2007/2007/42/42:94,135:3,215:146",DIAGN_N180,2.0,-0.071607,0.390020,0.029641,0.0,0.029641,2.0,0.019456,9.5,-0.264110
2,"r1/2007/2007/42/42:94,135:3,215:146",DIAGN_N180,5.0,0.164608,0.390020,0.029641,0.0,0.029641,2.0,0.019456,9.5,-0.264110
3,"r1/2007/2007/42/42:94,135:3,215:146",DIAGN_N180,7.6,0.369328,0.390020,0.029641,0.0,0.029641,2.0,0.019456,9.5,-0.264110
4,"r1/2007/2007/42/42:94,135:3,215:146",DIAGN_N180,11.0,0.240874,0.390020,0.029641,0.0,0.029641,2.0,0.019456,9.5,-0.264110
...,...,...,...,...,...,...,...,...,...,...,...,...
3354,"r1/2013/2013/42/42:273,135:220,215:35",DIAGN_N180,14.0,0.673137,0.207276,-0.008285,0.0,-0.008285,3.0,0.019337,12.0,0.340333
3355,"r1/2013/2013/42/42:273,135:220,215:35",DIAGN_I500,0.0,-0.002385,0.207276,0.191975,0.0,0.191975,NaN,NaN,NaN,NaN
3356,"r1/2013/2013/42/42:273,135:220,215:35",DIAGN_I500,1.0,0.381566,0.207276,0.191975,0.0,0.191975,NaN,NaN,NaN,NaN
3357,"r1/2013/2013/42/42:273,135:220,215:35",DIAGN_M815,0.0,-0.002385,0.207276,0.027425,0.0,0.027425,NaN,NaN,NaN,NaN


,concept,reference,year,leaf,conditions,fit_support,fit_activation,evaluation_support,evaluation_activation,evaluation_mae
0,"r1/2007/2007/42/42:94,135:3,215:146",2007,2007,5,EVENT_C1DIALISE_HD <= 11.5 and DIAGN_N180 <= 5...,57,0.245593,120,0.257936,0.272725
1,"r1/2007/2007/42/42:94,135:3,215:146",2007,2007,6,EVENT_C1DIALISE_HD <= 11.5 and DIAGN_N180 <= 5...,13,0.498870,20,0.505275,0.182404
2,"r1/2007/2007/42/42:94,135:3,215:146",2007,2007,4,EVENT_C1DIALISE_HD <= 11.5 and DIAGN_N180 > 5.5,18,1.440461,36,1.862395,0.846298
3,"r1/2007/2007/42/42:94,135:3,215:146",2007,2007,7,EVENT_C1DIALISE_HD > 11.5 and EVENT_c2MED_HIDF...,12,0.000000,18,0.000000,0.000000
4,"r1/2007/2007/42/42:94,135:3,215:146",2007,2007,8,EVENT_C1DIALISE_HD > 11.5 and EVENT_c2MED_HIDF...,19,0.000000,43,0.001223,0.001223
...,...,...,...,...,...,...,...,...,...,...
3644,"r1/2013/2013/42/42:273,135:220,215:35",2013,2014,2,EVENT_c3ACESSO_CT > 3,17,1.030265,36,1.386842,0.482379
3645,"r1/2013/2013/42/42:273,135:220,215:35",2013,2015,5,EVENT_c3ACESSO_CT <= 3 and EVENT_c2MED_CALCI <...,101,0.041919,169,1.003646,0.961727
3646,"r1/2013/2013/42/42:273,135:220,215:35",2013,2015,6,EVENT_c3ACESSO_CT <= 3 and EVENT_c2MED_CALCI <...,23,0.163370,48,0.827450,0.664080
3647,"r1/2013/2013/42/42:273,135:220,215:35",2013,2015,4,EVENT_c3ACESSO_CT <= 3 and EVENT_c2MED_CALCI >...,20,0.393280,56,1.027328,0.634100


,concept,reference,year,rule,clause,feature,operator,value,source
0,"r1/2007/2007/42/42:94,135:3,215:146",2007,2007,0,0,EVENT_C1DIALISE_HD,<=,0.032115,original_frozen_rule
1,"r1/2007/2007/42/42:94,135:3,215:146",2007,2007,0,1,EVENT_c2MED_ERITRO,>,5.410399,original_frozen_rule
2,"r1/2007/2007/42/42:94,135:3,215:146",2007,2007,0,2,DIAGN_N180,>,4.293990,original_frozen_rule
3,"r1/2007/2007/42/42:94,135:3,215:146",2007,2007,1,0,EVENT_C1DIALISE_HD,<=,0.087368,original_frozen_rule
4,"r1/2007/2007/42/42:94,135:3,215:146",2007,2007,1,1,DIAGN_N180,>,1.100664,original_frozen_rule
...,...,...,...,...,...,...,...,...,...
2026,"r1/2013/2013/42/42:273,135:220,215:35",2013,2013,0,1,EVENT_c3ACESSO_FV,<=,0.214957,original_frozen_rule
2027,"r1/2013/2013/42/42:273,135:220,215:35",2013,2013,0,0,EVENT_C1DIALISE_HD,<=,14.733785,same_year_description
2028,"r1/2013/2013/42/42:273,135:220,215:35",2013,2013,0,1,EVENT_c3ACESSO_CT,>,1.779635,same_year_description
2029,"r1/2013/2013/42/42:273,135:220,215:35",2013,2013,0,2,EVENT_c3ACESSO_FV,<=,0.623870,same_year_description


,aliases,concepts,kind,median_mae,median_normalized_delta,stratum
0,all_history,33,additive,0.265096,0.230199,full
1,all_history,33,constant_mean,0.655582,0.911006,full
2,all_history,33,constant_median,0.477732,0.676155,full
3,all_history,33,nonlinear,0.147428,0.000000,full
4,all_history,33,ridge,0.362596,0.428757,full
...,...,...,...,...,...,...
115,"last_3,all_history,baseline",4,constant_mean,0.427897,4.203227,zero
116,"last_3,all_history,baseline",4,constant_median,0.000000,-1.606393,zero
117,"last_3,all_history,baseline",4,nonlinear,0.178558,0.000000,zero
118,"last_3,all_history,baseline",4,ridge,0.234415,6.453275,zero


Report: frozen candidates, support, high-stratum errors, decisions, curves and tree groups


## Checks
Synthetic checks call the same helpers. They cover fitting-only transforms, additive sums, tree paths, threshold ties, zero-heavy high tails, protected evaluation inputs and historical benchmark settings. Check mode never loads real observations or writes scientific results.


### fit-only ranking excludes constants


In [16]:
checks = []
rng = np.random.default_rng(42)
X = rng.normal(size=(96, 4))
X[:,0] = 1
y = np.maximum(X[:,2], 0)*4 + X[:,1]*.2
ranking = rank_features(X, y)
assert ranking[0]["feature"] == 2 and 0 not in [m["feature"] for m in ranking]
checks.append("fit-only ranking excludes constants")
display(pd.DataFrame({"check": [checks[-1]], "result": ["passed"]}))

,check,result
0,fit-only ranking excludes constants,passed


### centered additive sum and frozen transforms


In [17]:
model = fit_additive(X, y, ranking, 3, .1)
transformed = (additive_basis(X, model["metadata"])-model["center"])/model["scale"]
np.testing.assert_allclose(predict_model(model, X), model["model"].intercept_+transformed @ model["model"].coef_)
np.testing.assert_allclose(transformed.mean(0), 0, atol=1e-14)
original = predict_model(model, X).copy()
poisoned = X.copy()
poisoned[:,2] += 1e6
predict_model(model, poisoned)
np.testing.assert_array_equal(predict_model(model, X), original)
checks.append("centered additive sum and frozen transforms")
display(pd.DataFrame({"check": [checks[-1]], "result": ["passed"]}))

,check,result
0,centered additive sum and frozen transforms,passed


### additive model has no interactions


In [18]:
corners = np.tile(X[0], (4, 1))
corners[1,1] += .3
corners[2,2] += .7
corners[3,1] += .3
corners[3,2] += .7
prediction = predict_model(model, corners)
np.testing.assert_allclose(prediction[3]-prediction[2]-prediction[1]+prediction[0], 0, atol=1e-12)
checks.append("additive model has no interactions")
display(pd.DataFrame({"check": [checks[-1]], "result": ["passed"]}))

,check,result
0,additive model has no interactions,passed


### fitting-only quartile knots


In [19]:
for m in model["metadata"]:
        np.testing.assert_array_equal(m["knots"], np.unique(np.quantile(X[:,m["feature"]], [.25,.5,.75])))
checks.append("fitting-only quartile knots")
display(pd.DataFrame({"check": [checks[-1]], "result": ["passed"]}))

,check,result
0,fitting-only quartile knots,passed


### strict cutoff, deterministic threshold, infeasible support


In [20]:
target = np.array([0,0,0,1,1], bool)
prediction = np.arange(5.)
threshold = tune_threshold(target, prediction)
assert threshold["threshold"] == 2 and threshold["search_size"] == 7
assert tune_threshold(np.ones(5, bool), prediction)["status"] == "unsupported"
assert tune_threshold(np.zeros(5, bool), prediction)["status"] == "unsupported"
assert tune_threshold(target, np.ones(5))["status"] == "unsupported"
checks.append("strict cutoff, deterministic threshold, infeasible support")
display(pd.DataFrame({"check": [checks[-1]], "result": ["passed"]}))

,check,result
0,"strict cutoff, deterministic threshold, infeas...",passed


### complexity ties favor fewer features


In [21]:
attempted = [{"model":"small", "kind":"additive", "status":"measured", "tune_mae":1., "features":3,"alpha":1.},
                 {"model":"large", "kind":"additive", "status":"measured", "tune_mae":1.-1e-7,"features":5,"alpha":10.}]
assert choose_complexity(attempted,"additive") == "small"
checks.append("complexity ties favor fewer features")
display(pd.DataFrame({"check": [checks[-1]], "result": ["passed"]}))

,check,result
0,complexity ties favor fewer features,passed


### tree paths, leaves and activation values


In [22]:
tree = DecisionTreeRegressor(max_depth=3, max_leaf_nodes=3, min_samples_leaf=10, random_state=SEED).fit(X,y)
groups = tree_groups({"model":tree}, ["constant","first","signal","noise"], X,y,"synthetic",2007,2010)
assert sum(g["evaluation_support"] for g in groups) == len(X)
assert all(g["conditions"] for g in groups)
assert all(g["reference"] == 2007 and g["year"] == 2010 for g in groups)
for group in groups:
        mask = tree.apply(X) == group["leaf"]
        np.testing.assert_allclose(tree.predict(X[mask]), group["fit_activation"])
checks.append("tree paths, leaves and activation values")
display(pd.DataFrame({"check": [checks[-1]], "result": ["passed"]}))

,check,result
0,"tree paths, leaves and activation values",passed


### zero-heavy high-tail and zero/empty denominators


In [23]:
truth = np.r_[np.zeros(90),np.full(10,10.)]
zero = np.zeros(100)
weights = np.ones((4,100))
full = continuous_record(truth,zero,truth,zero,weights)
high = continuous_record(truth[-10:],zero[-10:],truth[-10:],zero[-10:],weights[:,-10:])
assert full["mae"] == 1 and high["mae"] == 10
assert high["mse_low"] == high["mse_high"] == 100 and full["retained_reduction_low"] == 0
empty = continuous_record(np.array([]), np.array([]), np.array([]), np.array([]), np.empty((4,0)))
assert empty["mae"] is None and empty["status"] == "undefined"
exact = continuous_record(zero,zero,zero,zero,weights)
assert exact["normalized_delta"] is None and exact["retained_reduction"] is None
assert exact["normalized_delta_undefined_fraction"] == exact["retained_reduction_undefined_fraction"] == 1
assert exact["undefined_fraction"] == exact["mse_undefined_fraction"] == 0
checks.append("zero-heavy high-tail and zero/empty denominators")
display(pd.DataFrame({"check": [checks[-1]], "result": ["passed"]}))

,check,result
0,zero-heavy high-tail and zero/empty denominators,passed


### historical nonlinear and fixed ridge settings


In [24]:
_, booster, error = fit_candidate("nonlinear","nonlinear",X,y,ranking,None)
assert error is None and isinstance(booster["model"], HistGradientBoostingRegressor)
parameters = booster["model"].get_params()
assert all(parameters[k] == v for k,v in dict(max_iter=50,max_leaf_nodes=7,min_samples_leaf=10,l2_regularization=1.,early_stopping=False,random_state=42).items())
_, ridge, error = fit_candidate("ridge","ridge",X,y,ranking,None)
assert error is None and ridge["model"].steps[1][1].alpha == 1.
checks.append("historical nonlinear and fixed ridge settings")
display(pd.DataFrame({"check": [checks[-1]], "result": ["passed"]}))

,check,result
0,historical nonlinear and fixed ridge settings,passed


### 16-fit panel and evaluation-poisoning independence


In [25]:
fit, tune = np.arange(64), np.arange(64,96)
panel = fit_panel(X[fit],y[fit],X[tune],y[tune],1.)
assert len(panel["attempts"]) == 16 and len(panel["models"]) == 16
assert panel["selected"]["additive"] in panel["models"] and panel["selected"]["tree"] in panel["models"]
selected = panel["selected"].copy()
predict_model(panel["models"][selected["additive"]],poisoned)
assert panel["selected"] == selected
checks.append("16-fit panel and evaluation-poisoning independence")
display(pd.DataFrame({"check": [checks[-1]], "result": ["passed"]}))

,check,result
0,16-fit panel and evaluation-poisoning independ...,passed


### four actual-activation strata, three-variant aliases, binary-only rules and fixed schemas


In [26]:
weights = np.ones((4,len(tune)))
fixture = {"key":"synthetic", "reference":2007, "aliases":["baseline", "last_3", "all_history"],
               "X":X[tune], "activation":y[tune], "patients":np.arange(len(tune)), "cutoff":1., "rule_mask":y[tune]>1.}
continuous, decisions = score_year(fixture,panel,np.arange(len(tune)),weights,2007)
assert len(continuous) == 16*4 and any(r["model"] == "original_rule" for r in decisions)
assert all(set(r) <= set(CONTINUOUS_FIELDS) for r in continuous)
assert all(set(r) <= set(BINARY_FIELDS) for r in decisions)
assert {r["stratum"] for r in continuous} == {"full", "zero", "positive", "original_high"}
assert all("mae" not in r for r in decisions)
checks.append("four actual-activation strata, three-variant aliases, binary-only rules and fixed schemas")
display(pd.DataFrame({"check": [checks[-1]], "result": ["passed"]}))

,check,result
0,"four actual-activation strata, three-variant a...",passed


### weighted binary counts and zero bootstrap support


In [27]:
sampled = sampled_binary(np.array([0,1]),np.array([0,1]),np.array([[1,2],[0,0]]))
assert sampled["f2"][0] == 1 and np.isnan(sampled["f2"][1])
checks.append("weighted binary counts and zero bootstrap support")
display(pd.DataFrame({"check": [checks[-1]], "result": ["passed"]}))

,check,result
0,weighted binary counts and zero bootstrap support,passed


### constant-feature failures and absent-target thresholds remain visible


In [28]:
constant_panel = fit_panel(np.ones((32,2)),np.zeros(32),np.ones((12,2)),np.zeros(12),.1)
assert len(constant_panel["attempts"]) == 16
assert sum(a["status"] == "failed" for a in constant_panel["attempts"]) == 9
assert constant_panel["selected"]["additive"] is None
assert all(t["status"] == "unsupported" for t in constant_panel["thresholds"].values())
checks.append("constant-feature failures and absent-target thresholds remain visible")
display(pd.DataFrame({"check": [checks[-1]], "result": ["passed"]}))

,check,result
0,constant-feature failures and absent-target th...,passed


### aggregate export validates actual scoring records


In [29]:
from cmr_common import public_records
public_records(continuous, CONTINUOUS_FIELDS)
public_records(decisions, BINARY_FIELDS)
public_records(panel["attempts"], ATTEMPT_FIELDS)
checks.append("aggregate export validates actual scoring records")
display(pd.DataFrame({"check": [checks[-1]], "result": ["passed"]}))

,check,result
0,aggregate export validates actual scoring records,passed


### missing benchmarks retain raw errors and original-rule fidelity


In [30]:
partial = {**panel, "models": {name:bundle for name,bundle in panel["models"].items() if name not in ("nonlinear", "constant_median")}}
partial_continuous, partial_decisions = score_year(fixture,partial,np.arange(len(tune)),weights,2007)
assert len(partial_continuous) == 14*4, "Valid model errors must survive missing benchmarks"
assert any(r["model"] == "original_rule" for r in partial_decisions), "Original R fidelity must survive missing benchmarks"
assert all(r["normalized_delta"] is None for r in partial_continuous)
assert any(r["mae"] is not None for r in partial_continuous)
checks.append("missing benchmarks retain raw errors and original-rule fidelity")
display(pd.DataFrame({"check": [checks[-1]], "result": ["passed"]}))

,check,result
0,missing benchmarks retain raw errors and origi...,passed


In [31]:
CHECKS = pd.DataFrame({"check":checks, "result":"passed"})
display(CHECKS)
assert CHECKS.result.eq("passed").all()
print(f"{len(CHECKS)} explanation checks passed")
if MODE != "check":
    finish_run(CTX, "completed", {"concepts": len(BUNDLES), "continuous_rows": len(CONTINUOUS), "binary_rows": len(BINARY)})

,check,result
0,fit-only ranking excludes constants,passed
1,centered additive sum and frozen transforms,passed
2,additive model has no interactions,passed
3,fitting-only quartile knots,passed
4,"strict cutoff, deterministic threshold, infeas...",passed
5,complexity ties favor fewer features,passed
6,"tree paths, leaves and activation values",passed
7,zero-heavy high-tail and zero/empty denominators,passed
8,historical nonlinear and fixed ridge settings,passed
9,16-fit panel and evaluation-poisoning independ...,passed


15 explanation checks passed
